# Hallmark Python Demo: init, info, add, commit, checkout, status, and clone


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from hallmark import Repo
from hallmark.downloader import download_remote_data
from hallmark.error import DestinationExistsError


In [2]:
%%bash
rm -rf repo1
rm -rf hallmark-demo-clone
mkdir repo1


## 1. Initialize the hallmark repo


In [3]:
repo1 = Repo.init("repo1")


## 2. Create data files on `main`


In [4]:
%%bash
pushd repo1
for f in a{0,0.75}_i{0,30,60,90,120}.h5; do echo "$f" > "$f"; done
echo "Files in repo1/:"
ls *.h5
popd


~/astro_se_int/hallmark/demo/repo1 ~/astro_se_int/hallmark/demo
Files in repo1/:
a0_i0.h5
a0_i120.h5
a0_i30.h5
a0_i60.h5
a0_i90.h5
a0.75_i0.h5
a0.75_i120.h5
a0.75_i30.h5
a0.75_i60.h5
a0.75_i90.h5
~/astro_se_int/hallmark/demo


## 3. Add files and commit on `main`


In [5]:
paraframe = repo1.add("a{a}_i{i}.h5")
paraframe

,path,a,i
0,a0.75_i0.h5,0.75,0.0
1,a0.75_i120.h5,0.75,120.0
2,a0.75_i30.h5,0.75,30.0
3,a0.75_i60.h5,0.75,60.0
4,a0.75_i90.h5,0.75,90.0
5,a0_i0.h5,0.00,0.0
6,a0_i120.h5,0.00,120.0
7,a0_i30.h5,0.00,30.0
8,a0_i60.h5,0.00,60.0
9,a0_i90.h5,0.00,90.0


This paraframe object will also be reflected in the data.tsv file.

In [6]:
repo1.status()

{'branch': 'main',
 'staged': {'state': ['config.yml', 'data.tsv', 'meta.yml'],
  'added': ['a0.75_i0.h5',
   'a0.75_i120.h5',
   'a0.75_i30.h5',
   'a0.75_i60.h5',
   'a0.75_i90.h5',
   'a0_i0.h5',
   'a0_i120.h5',
   'a0_i30.h5',
   'a0_i60.h5',
   'a0_i90.h5'],
  'modified': [],
  'deleted': []},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

In [7]:
repo1.commit("add main dataset")


True

In [8]:
repo1.status()

{'branch': 'main',
 'staged': {'state': [], 'added': [], 'modified': [], 'deleted': []},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

In [9]:
%%bash
echo "Objects stored after commit:"
find repo1/.hm/objects -type f | wc -l


Objects stored after commit:
      10


### Commit log


In [10]:
print(repo1.log())


commit d3da52e3a3077ce58995037957d74d1316fadde8
Author: Nayera Abdessalam <nayeramarie16@gmail.com>
Date:   Fri May 1 09:27:14 2026 -0700

    add main dataset

commit ecb6fc7ffa7fd6b61d59a580d131c829caf942a8
Author: Nayera Abdessalam <nayeramarie16@gmail.com>
Date:   Fri May 1 09:27:14 2026 -0700

    Initial commit: local `.hm` repository


## 4. Demo for `repo.checkout('BRANCH')`

We use a temporary branch so the main workflow stays intact.


In [11]:
# add(".") rebuilds the manifest from files that currently exist

repo1.checkout("experiment")

for path in Path(str(repo1.worktree)).glob("a*.h5"):
    path.unlink()

b_files = [
    f"b{a}_i{i}.h5"
    for a in [0, 0.75]
    for i in [0, 30, 60, 90, 120]
    ]

for name in b_files:
    path = Path(str(repo1.worktree)) / name
    path.write_text(f"{name}\n", encoding="utf-8")

repo1.set_config(fmt="b{a}_i{i}.h5")
repo1.add(".")
repo1.commit("add experiment dataset")

repo1.status()


{'branch': 'experiment',
 'staged': {'state': [], 'added': [], 'modified': [], 'deleted': []},
 'worktree': {'modified': [], 'deleted': []},
 'untracked': []}

In [12]:
# Check back to main to see that the manifest has been correctly rebuilt
repo1.checkout("main")

True

In [13]:
# Show available branches
repo1.branches()

{'current': 'main', 'names': ['experiment', 'main']}

In [14]:
%%bash
echo "Files after checkout back to main:"
ls repo1/*.h5


Files after checkout back to main:
repo1/a0_i0.h5
repo1/a0_i120.h5
repo1/a0_i30.h5
repo1/a0_i60.h5
repo1/a0_i90.h5
repo1/a0.75_i0.h5
repo1/a0.75_i120.h5
repo1/a0.75_i30.h5
repo1/a0.75_i60.h5
repo1/a0.75_i90.h5


## 5. Filter Functionality


We can also show that we can filter data from the ParaFrame Object

In [15]:
# Single Parameter Filter
paraframe(a=0)

,path,a,i
5,a0_i0.h5,0.0,0.0
6,a0_i120.h5,0.0,120.0
7,a0_i30.h5,0.0,30.0
8,a0_i60.h5,0.0,60.0
9,a0_i90.h5,0.0,90.0


In [16]:
# Or Filter
paraframe(a=0, i=30)

,path,a,i
2,a0.75_i30.h5,0.75,30.0
5,a0_i0.h5,0.00,0.0
6,a0_i120.h5,0.00,120.0
7,a0_i30.h5,0.00,30.0
8,a0_i60.h5,0.00,60.0
9,a0_i90.h5,0.00,90.0


In [17]:
# And Filter
paraframe(a=0)(i=30)

,path,a,i
7,a0_i30.h5,0.0,30.0


In [18]:
# Masking Filter retains Pandas DataFrame functionality
paraframe[(30 <= paraframe.i) & (paraframe.i <= 100)]

,path,a,i
2,a0.75_i30.h5,0.75,30.0
3,a0.75_i60.h5,0.75,60.0
4,a0.75_i90.h5,0.75,90.0
7,a0_i30.h5,0.00,30.0
8,a0_i60.h5,0.00,60.0
9,a0_i90.h5,0.00,90.0


## 6. Clone Functionality


In [27]:
# Demo: clone a remote demo repository
from hallmark.error import DestinationExistsError

try:
    clone = Repo.clone('https://github.com/l6a/hallmark-demo-repo.hm.git', "hallmark-demo-clone", show_progress=True)
    clone_result = {
        "dot_hallmark_repo": str(clone.dothm.path),
        "hallmark_worktree": str(clone.worktree),
        "branches": clone.branches(),
        "data_rows": len(clone.state.data),
    }
except DestinationExistsError as exc:
    clone_result = str(exc)

clone_result


100%|██████████| 8/8 [00:03<00:00,  2.26file/s]


DownloadError: Failed to download 8 file(s):
  - Checksum mismatch for SR1_M87_2017_095_hi_hops_netcal_StokesI.uvfits.part
  - Checksum mismatch for SR1_M87_2017_096_lo_hops_netcal_StokesI.uvfits.part
  - Checksum mismatch for SR1_M87_2017_095_lo_hops_netcal_StokesI.uvfits.part
  - Checksum mismatch for SR1_M87_2017_101_hi_hops_netcal_StokesI.uvfits.part
  - Checksum mismatch for SR1_M87_2017_096_hi_hops_netcal_StokesI.uvfits.part
  - ... 3 more error(s)